# Nudged Elastic Band: Minimum Energy Reaction Path

**Pipeline**: Reactant & product geometries → Interpolated images → Hamiltonian per image → Eigensolve (uniqx) → NEB force projection → Path optimisation

The **Nudged Elastic Band** (NEB) method finds the **minimum energy path** (MEP)
between a reactant state and a product state on a potential energy surface. It is
the standard tool for locating **transition states** — the saddle points that
govern reaction rates through the Arrhenius equation:

$$k = A \exp\!\left(-\frac{E_a}{k_B T}\right)$$

NEB works by threading a chain of **images** (intermediate geometries) between
reactant and product, connected by harmonic spring forces. At each iteration:

1. Compute the energy $E_i$ and true force $\mathbf{F}_i^{\text{true}} = -\nabla E_i$ at each image.
2. Estimate the path **tangent** $\hat{\boldsymbol{\tau}}_i$ at each image.
3. Project out the component of the true force parallel to the path:
 $$\mathbf{F}_i^\perp = \mathbf{F}_i^{\text{true}} - (\mathbf{F}_i^{\text{true}} \cdot \hat{\boldsymbol{\tau}}_i)\,\hat{\boldsymbol{\tau}}_i$$
4. Add the spring force along the tangent to maintain equal image spacing:
 $$\mathbf{F}_i^{\text{spring}} = k(|\mathbf{r}_{i+1} - \mathbf{r}_i| - |\mathbf{r}_i - \mathbf{r}_{i-1}|)\,\hat{\boldsymbol{\tau}}_i$$
5. The total NEB force is $\mathbf{F}_i^{\text{NEB}} = \mathbf{F}_i^\perp + \mathbf{F}_i^{\text{spring}}$.

The key insight: by removing the parallel component of the true force and adding
the parallel spring force, images converge to the MEP without sliding down to
the endpoints or bunching up.

We demonstrate this on a 1D **S$_\text{N}$2 reaction model**:

$$\text{Cl}^- + \text{CH}_3\text{Cl} \rightarrow \text{ClCH}_3 + \text{Cl}^-$$

modelled by a double-well potential $E(s) = as^4 - bs^2 + c$ along the reaction
coordinate $s$. The transition state (TS) sits at the saddle point $s = 0$.
Each image's energy is obtained by diagonalising a Hamiltonian $H(s)$ on the
uniqx via `ops.eigs`.

| Step | Key op | Hardware |
|:-----|:-------|:---------|
| Energy per image | `eigs` (dim 4–16) | CPU / GPU |
| NEB forces | Finite differences (client) | Local |
| Path update | Steepest descent (client) | Local |

## 1. Setup & Configuration

In [ ]:
# Copyright (c) 2024-2026 ORIQX AG/LTD/SAS. All rights reserved.
# NEB reaction path example notebook.

import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)
import math
import time
import numpy as np
import matplotlib.pyplot as plt

from uniqx import ops, tracing, fmt_mat, parse_result
from uniqx.ops import SX, SZ, I2
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")

# ----- NEB parameters -----
N_IMAGES = 7         # Number of intermediate images (excluding endpoints)
N_QUBITS = 2         # Active-space qubits for the Hamiltonian
K_SPRING = 1.0       # Spring constant for inter-image springs (eV/A^2)
STEP_SIZE = 0.05     # Steepest-descent step size for image update
N_ITER = 10          # Number of NEB iterations
FORCE_TOL = 0.01     # Convergence threshold on max |F_NEB|

# Double-well PES parameters: E(s) = a*s^4 - b*s^2 + c
PES_A = 0.1    # quartic coefficient (eV)
PES_B = 1.0    # quadratic coefficient (eV)
PES_C = 2.5    # energy offset (eV)

# Reactant and product in reaction coordinate
S_REACTANT = -2.0
S_PRODUCT = 2.0

print(f"NEB: {N_IMAGES} images, k_spring={K_SPRING}, {N_ITER} max iterations")
print(f"PES: E(s) = {PES_A}*s^4 - {PES_B}*s^2 + {PES_C}")
print(f"Endpoints: s_reactant={S_REACTANT}, s_product={S_PRODUCT}")


## 2. Reaction Model: SN2 Double-Well Potential

The S$_\text{N}$2 reaction Cl$^-$ + CH$_3$Cl $\rightarrow$ ClCH$_3$ + Cl$^-$
is modelled by a symmetric double-well potential along the reaction coordinate $s$:

$$E(s) = a\,s^4 - b\,s^2 + c$$

The minima (reactant and product wells) sit at $s = \pm\sqrt{b/2a}$ and the
**transition state** is the local maximum at $s = 0$ with barrier height
$E_\text{TS} - E_\text{min}$.

We map this to a quantum Hamiltonian $H(s)$ whose ground-state eigenvalue
reproduces $E(s)$. This is the energy that uniqx computes for each NEB
image.

In [ ]:
# --- Pure-Python matrix helpers (used in traced IR, no numpy) ---


def kron(A, B):
    """Kronecker product of two matrices."""
    rA, rB = len(A), len(B)
    n = rA * rB
    C = [[0.0] * n for _ in range(n)]
    for i in range(rA):
        for j in range(rA):
            for k in range(rB):
                for l in range(rB):
                    C[i * rB + k][j * rB + l] = A[i][j] * B[k][l]
    return C


def eye(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def matadd(A, B):
    return [[A[i][j] + B[i][j] for j in range(len(A))] for i in range(len(A))]


def matscale(s, A):
    return [[s * x for x in row] for row in A]


def embed_local(op, qubit, n_qubits):
    result = eye(1)
    for k in range(n_qubits):
        result = kron(result, op if k == qubit else I2)
    return result


def two_body_local(opA, opB, qa, qb, n_qubits):
    parts = []
    for k in range(n_qubits):
        if k == qa:
            parts.append(opA)
        elif k == qb:
            parts.append(opB)
        else:
            parts.append(I2)
    result = eye(1)
    for part in parts:
        result = kron(result, part)
    return result


def pes_energy(s):
    """Analytical double-well energy at reaction coordinate s."""
    return PES_A * s**4 - PES_B * s**2 + PES_C


def build_reaction_hamiltonian(s, n_qubits):
    """Build a Hamiltonian whose ground-state energy encodes E(s).

    The Hamiltonian is:
      H(s) = E(s) * I  +  delta(s) * Z0  +  coupling(s) * X0  +  ZZ interactions

    where:
      E(s)       = a*s^4 - b*s^2 + c         (the PES energy)
      delta(s)   = 0.5 * exp(-s^2)            (splitting near TS)
      coupling(s)= 0.2 * exp(-0.5*(s/1.5)^2)  (off-diagonal coupling)

    The ground eigenvalue closely tracks E(s), with small corrections from
    the coupling terms that make the eigensolve non-trivial.
    """
    dim = 2 ** n_qubits
    e_s = PES_A * s**4 - PES_B * s**2 + PES_C

    # Level splitting: largest near the transition state (s=0)
    delta = 0.5 * math.exp(-s * s)

    # Off-diagonal coupling: Gaussian centred at TS
    coupling = 0.2 * math.exp(-0.5 * (s / 1.5) ** 2)

    # H = e_s * I + delta * Z0 + coupling * X0
    H = matscale(e_s, eye(dim))
    H = matadd(H, matscale(delta, embed_local(SZ, 0, n_qubits)))
    H = matadd(H, matscale(coupling, embed_local(SX, 0, n_qubits)))

    # Add weak ZZ interaction between qubits for richer structure
    for q in range(1, n_qubits):
        zz_strength = 0.03 * math.exp(-0.5 * (s / 2.0) ** 2) / q
        H = matadd(H, matscale(zz_strength, two_body_local(SZ, SZ, 0, q, n_qubits)))

    return H, dim


# Test the Hamiltonian at key points
for s_test in [S_REACTANT, 0.0, S_PRODUCT]:
    H_t, dim = build_reaction_hamiltonian(s_test, N_QUBITS)
    e_analytical = pes_energy(s_test)
    print(f"s={s_test:+5.1f}: E_analytical={e_analytical:.4f} eV, dim={dim}")

# PES properties
s_min = math.sqrt(PES_B / (2.0 * PES_A))
E_min = pes_energy(s_min)
E_ts = pes_energy(0.0)
barrier = E_ts - E_min
print(f"\nPES minima at s = +/-{s_min:.3f}")
print(f"E_min = {E_min:.4f} eV")
print(f"E_TS  = {E_ts:.4f} eV")
print(f"Barrier height = {barrier:.4f} eV")

## 3. Initial Path: Linear Interpolation

Create the initial NEB chain by linearly interpolating $N$ images between the
reactant ($s = -2$) and product ($s = +2$). The endpoints are fixed and never
move during optimisation.

The initial path is a straight line in reaction-coordinate space, which is far
from the MEP — images near $s = 0$ sit at the transition state barrier.

In [ ]:
# Linear interpolation: N_IMAGES intermediate images + 2 fixed endpoints
n_total = N_IMAGES + 2  # total including endpoints
images = np.linspace(S_REACTANT, S_PRODUCT, n_total)

print(f"Initial chain: {n_total} images (2 fixed endpoints + {N_IMAGES} movable)")
print(f"Image positions: {', '.join(f'{s:.3f}' for s in images)}")
print(f"\nInitial energies (analytical):")
for i, s in enumerate(images):
    tag = "[fixed]" if i == 0 or i == n_total - 1 else ""
    print(f"  Image {i}: s={s:+6.3f}, E={pes_energy(s):.4f} eV {tag}")

## 4. Trace the Eigensolve Module

For each image, we diagonalise $H(s_i)$ to get the ground-state energy.
The traced module uses `ops.eigs` with `k=1` to request just the lowest
eigenvalue. uniqx routes this to LAPACK (CPU) or cuSOLVER (GPU)
depending on matrix size and available hardware.

In [ ]:
dim = 2 ** N_QUBITS


@tracing.to_module(name="neb_eigsolve")
def neb_eigsolve(H_mat):
    """Compute the ground-state eigenvalue of the reaction Hamiltonian."""
    eigenvalues, eigenvectors = ops.eigs(
        H_mat, k=1, hermitian=True, which="smallest"
    )
    return eigenvalues, eigenvectors


# Trace with test Hamiltonian at s=0
H_test, _ = build_reaction_hamiltonian(0.0, N_QUBITS)
mod_test = neb_eigsolve(H_test)
ir_text = mod_test.to_text()
n_ops = len(mod_test.functions[0].ops)
n_params = len(mod_test.functions[0].params)

print(f"NEB eigsolve module: {n_ops} ops, {n_params} params")
print(f"Requesting k=1 eigenvalue (ground state only)")
print(f"Matrix dimension: {dim}x{dim}")
print(f"IR size: {len(ir_text)} chars")
print(f"\nFirst 400 chars of IR:\n{ir_text[:400]}...")

## 5. Preflight: Execution Options

Query uniqx for Pareto-optimal hardware assignments. For a small
$4 \times 4$ Hamiltonian, CPU is typically cheapest. As the system grows
(more qubits), GPU and QPU options become competitive.

Each NEB iteration submits $N$ independent eigensolves — these can be
parallelised across the available hardware fleet.

In [ ]:
options = preflight(mod_test, client=client)
print(f"Pre-flight options (job_id={options.job_id}):\n")

for opt in options:
    rec = " <-- recommended" if opt.get("recommended") else ""
    print(
        f"  {opt['label']:<20} "
        f"time={opt['total_time']:>8.2f}s  "
        f"cost=${opt['total_cost_usd']:.6f}  "
        f"err={opt['max_error_rate']:.4f}"
        f"{rec}"
    )

# Select recommended option
rec = options.recommended
SELECTED_OPTION = rec["_idx"]
print(f"\nSelected: {rec['label']} (option {SELECTED_OPTION})")
print(f"Per-image cost: ${rec['total_cost_usd']:.6f}")
print(f"Total estimated cost for {N_IMAGES} images x {N_ITER} iterations: "
      f"${rec['total_cost_usd'] * N_IMAGES * N_ITER:.4f}")

## 6. Energy Calculation: Submit All Images

Compute the ground-state energy at each image position by building $H(s_i)$,
tracing the eigensolve module, and submitting to uniqx.

We define a helper that submits all images and collects energies.

In [ ]:
def compute_energies(positions):
    """Submit eigensolves for all image positions and return energies.

    If a job fails (empty payload, missing eigenvalues, or runtime error),
    the corresponding entry is set to NaN so callers can decide how to
    handle missing images without aborting the full sweep.
    """
    energies = np.full(len(positions), np.nan)

    for i, s in enumerate(positions):
        try:
            H_s, dim_h = build_reaction_hamiltonian(s, N_QUBITS)
            mod = neb_eigsolve(H_s)
            jid = submit(
                mod,
                client=client,
                runtime_inputs=[fmt_mat(H_s, dim_h, dim_h)],
                preflight_job_id=options.job_id,
                option_idx=SELECTED_OPTION,
            )
            res = get(jid, client=client)

            payload = res.get("payload", b"")
            if isinstance(payload, str):
                payload = payload.encode()
            if not payload:
                print(f"  [skip] image {i} (s={s:+.3f}): empty payload")
                continue

            out = parse_result(payload, ["eigenvalues"])
            if not out or "eigenvalues" not in out:
                print(f"  [skip] image {i} (s={s:+.3f}): no eigenvalues in result")
                continue

            evals = out["eigenvalues"][2]
            if evals is None or len(evals) == 0:
                print(f"  [skip] image {i} (s={s:+.3f}): empty eigenvalue array")
                continue

            energies[i] = evals[0]  # ground-state energy
        except Exception as exc:
            print(f"  [skip] image {i} (s={s:+.3f}): {type(exc).__name__}: {exc}")
            continue

    return energies


def fill_missing_energies(positions, energies):
    """Fill NaN entries via linear interpolation from neighbours, falling
    back to the analytical PES at the endpoints. Keeps downstream NEB
    arithmetic finite when individual eigensolves fail."""
    energies = np.asarray(energies, dtype=float).copy()
    n = len(energies)
    if n == 0:
        return energies
    mask = np.isnan(energies)
    if not mask.any():
        return energies
    if mask.all():
        return np.array([pes_energy(s) for s in positions])
    good_idx = np.where(~mask)[0]
    bad_idx = np.where(mask)[0]
    energies[bad_idx] = np.interp(bad_idx, good_idx, energies[good_idx])
    return energies


# Compute initial energies
E_initial_raw = compute_energies(images)
n_missing = int(np.sum(np.isnan(E_initial_raw)))
if n_missing:
    print(f"\n[warn] {n_missing}/{len(images)} eigensolves returned no result; "
          f"interpolating from neighbours")
E_initial = fill_missing_energies(images, E_initial_raw)

print("\nInitial image energies:")
for i, (s, e, e_raw) in enumerate(zip(images, E_initial, E_initial_raw)):
    e_exact = pes_energy(s)
    tag = "[fixed]" if i == 0 or i == n_total - 1 else ""
    suffix = " (interpolated)" if np.isnan(e_raw) else ""
    print(f"  Image {i}: s={s:+6.3f}, E_gw={e:.4f}, E_exact={e_exact:.4f} eV {tag}{suffix}")

print(f"\nMax energy (initial TS estimate): {max(E_initial):.4f} eV at "
      f"s={images[np.argmax(E_initial)]:+.3f}")

## 7. NEB Forces

The NEB force on each image has two components:

### True force (perpendicular to path)

The true force from the PES is $F_i^{\text{true}} = -dE/ds$, estimated via
central finite differences:

$$F_i^{\text{true}} \approx -\frac{E(s_i + \delta) - E(s_i - \delta)}{2\delta}$$

We project out the component parallel to the path tangent:

$$F_i^\perp = F_i^{\text{true}} - (F_i^{\text{true}} \cdot \hat{\tau}_i)\,\hat{\tau}_i$$

### Spring force (parallel to path)

The spring force maintains equal spacing between images:

$$F_i^{\text{spring}} = k\,(s_{i+1} - 2s_i + s_{i-1})\,\hat{\tau}_i$$

### Total NEB force

$$F_i^{\text{NEB}} = F_i^\perp + F_i^{\text{spring}}$$

In 1D, the tangent is simply $\hat{\tau}_i = \text{sign}(s_{i+1} - s_{i-1})$,
so the projection is straightforward.

In [ ]:
def compute_neb_forces(positions, energies, k_spring, delta=0.01):
    """Compute NEB forces on all movable images.

    Returns:
        forces: array of NEB forces on images 1..N (excluding endpoints)
        f_true_perp: perpendicular true forces
        f_spring: spring forces along tangent
    """
    n = len(positions)
    forces = np.zeros(n)
    f_true_perp = np.zeros(n)
    f_spring_arr = np.zeros(n)

    for i in range(1, n - 1):  # skip fixed endpoints
        s_i = positions[i]

        # --- True force via finite differences on uniqx energies ---
        # We use the energies of neighbouring images for a cheaper estimate.
        # For higher accuracy, one could submit additional eigensolves at
        # s_i +/- delta, but using neighbours is standard NEB practice.
        dE_ds = (energies[i + 1] - energies[i - 1]) / (positions[i + 1] - positions[i - 1])
        f_true = -dE_ds

        # --- Path tangent (1D: just the sign of forward direction) ---
        tau = 1.0 if positions[i + 1] > positions[i - 1] else -1.0

        # --- Project out parallel component of true force ---
        f_parallel = f_true * tau  # scalar projection in 1D
        f_perp = f_true - f_parallel * tau

        # --- Spring force along tangent ---
        f_spring = k_spring * (positions[i + 1] - 2.0 * positions[i] + positions[i - 1]) * tau

        # --- Total NEB force ---
        forces[i] = f_perp + f_spring
        f_true_perp[i] = f_perp
        f_spring_arr[i] = f_spring

    return forces, f_true_perp, f_spring_arr


# Test on initial chain
F_neb, F_perp, F_spr = compute_neb_forces(images, E_initial, K_SPRING)

print("Initial NEB forces:")
print(f"  {'Image':>5} {'s':>8} {'F_NEB':>10} {'F_perp':>10} {'F_spring':>10}")
print(f"  {'-'*5:>5} {'-'*8:>8} {'-'*10:>10} {'-'*10:>10} {'-'*10:>10}")
for i in range(1, n_total - 1):
    print(f"  {i:>5} {images[i]:>+8.3f} {F_neb[i]:>+10.4f} {F_perp[i]:>+10.4f} {F_spr[i]:>+10.4f}")

print(f"\nMax |F_NEB| = {max(abs(F_neb[1:-1])):.4f} eV/A")

## 8. NEB Optimisation Loop

Iterate steepest-descent updates on the movable images using the NEB forces.
At each iteration:

1. Compute energies at all image positions (uniqx eigensolves).
2. Compute NEB forces.
3. Update movable image positions: $s_i \leftarrow s_i + \alpha \, F_i^{\text{NEB}}$.
4. Check convergence: $\max|F_i^{\text{NEB}}| < \text{tol}$.

We store the path at each iteration for visualisation.

In [ ]:
# Store history for plotting
path_history = [images.copy()]
energy_history = [E_initial.copy()]
max_force_history = [max(abs(F_neb[1:-1]))]

current_images = images.copy()
converged = False

print(f"{'Iter':>4} {'Max|F|':>10} {'TS energy':>12} {'TS position':>12} {'Status':>10}")
print(f"{'-'*4:>4} {'-'*10:>10} {'-'*12:>12} {'-'*12:>12} {'-'*10:>10}")

# Initial status
ts_idx = np.argmax(E_initial)
print(f"   0 {max_force_history[0]:>10.4f} {E_initial[ts_idx]:>12.4f} {current_images[ts_idx]:>+12.3f} {'initial':>10}")

t0_total = time.monotonic()

for iteration in range(1, N_ITER + 1):
    t0_iter = time.monotonic()

    # 1. Compute energies (fill any failed eigensolves from neighbours)
    E_current_raw = compute_energies(current_images)
    n_miss = int(np.sum(np.isnan(E_current_raw)))
    if n_miss:
        print(f"  iter {iteration}: {n_miss} eigensolves missing; interpolating")
    E_current = fill_missing_energies(current_images, E_current_raw)

    # 2. Compute NEB forces
    F_neb, F_perp, F_spr = compute_neb_forces(current_images, E_current, K_SPRING)

    # 3. Update movable images (steepest descent)
    for i in range(1, n_total - 1):
        current_images[i] += STEP_SIZE * F_neb[i]

    # 4. Record and check convergence
    max_force = max(abs(F_neb[1:-1]))
    ts_idx = np.argmax(E_current)

    path_history.append(current_images.copy())
    energy_history.append(E_current.copy())
    max_force_history.append(max_force)

    dt_iter = time.monotonic() - t0_iter
    status = "converged" if max_force < FORCE_TOL else f"{dt_iter:.1f}s"

    print(
        f"{iteration:>4} {max_force:>10.4f} {E_current[ts_idx]:>12.4f} "
        f"{current_images[ts_idx]:>+12.3f} {status:>10}"
    )

    if max_force < FORCE_TOL:
        converged = True
        break

dt_total = time.monotonic() - t0_total
n_submits = (N_ITER if not converged else iteration) * n_total
print(f"\nNEB {'converged' if converged else 'reached max iterations'} "
      f"in {dt_total:.2f}s ({n_submits} eigensolve submissions)")

## 9. Final Path: Compute Converged Energies

Recompute energies on the final converged (or last-iteration) image positions
to get the definitive MEP.

In [ ]:
E_final_raw = compute_energies(current_images)
E_final = fill_missing_energies(current_images, E_final_raw)
n_final_missing = int(np.sum(np.isnan(E_final_raw)))
if n_final_missing:
    print(f"[warn] {n_final_missing}/{len(current_images)} final eigensolves "
          f"unavailable; reported values are interpolated\n")

print("Final MEP:")
print(f"  {'Image':>5} {'s':>8} {'E (eV)':>10} {'E_exact':>10} {'Delta':>8}")
print(f"  {'-'*5:>5} {'-'*8:>8} {'-'*10:>10} {'-'*10:>10} {'-'*8:>8}")
for i, (s, e, e_raw) in enumerate(zip(current_images, E_final, E_final_raw)):
    e_exact = pes_energy(s)
    delta = abs(e - e_exact)
    is_ts = i == int(np.argmax(E_final))
    tag = " [TS]" if is_ts else ""
    if np.isnan(e_raw):
        tag += " (interp)"
    print(f"  {i:>5} {s:>+8.3f} {e:>10.4f} {e_exact:>10.4f} {delta:>8.4f}{tag}")

# Transition state analysis
ts_idx_final = int(np.argmax(E_final))
s_ts = current_images[ts_idx_final]
E_ts_neb = E_final[ts_idx_final]
E_react = E_final[0]
E_prod = E_final[-1]
barrier_fwd = E_ts_neb - E_react
barrier_rev = E_ts_neb - E_prod

print(f"\nTransition State Analysis:")
print(f"  TS position:    s = {s_ts:+.4f}")
print(f"  TS energy:      E = {E_ts_neb:.4f} eV")
print(f"  Forward barrier:  {barrier_fwd:.4f} eV")
print(f"  Reverse barrier:  {barrier_rev:.4f} eV")
print(f"  Exact TS energy:  {pes_energy(0.0):.4f} eV (at s=0)")
print(f"  |s_TS - 0|:       {abs(s_ts):.4f} (deviation from exact TS)")

## 10. Visualisation

Four diagnostic plots:

1. **Initial vs final MEP** overlaid on the true PES.
2. **Path evolution** across NEB iterations.
3. **Convergence** of the maximum NEB force.
4. **Barrier comparison** — NEB result vs exact PES vs direct scan.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Dense PES for reference curve
s_dense = np.linspace(S_REACTANT - 0.5, S_PRODUCT + 0.5, 200)
E_dense = np.array([pes_energy(s) for s in s_dense])

# --- Top-left: Initial vs final MEP ---
ax = axes[0, 0]
ax.plot(s_dense, E_dense, "-", color="#94a3b8", linewidth=1.5, label="Exact PES", zorder=1)
ax.plot(
    path_history[0], energy_history[0],
    "o--", color="#ef4444", markersize=6, linewidth=1.2, label="Initial path", zorder=3
)
ax.plot(
    current_images, E_final,
    "s-", color="#2563eb", markersize=7, linewidth=2, label="Converged MEP", zorder=4
)
# Mark TS
ax.plot(s_ts, E_ts_neb, "*", color="#f59e0b", markersize=18, zorder=5, label=f"TS (s={s_ts:+.2f})")
ax.set_xlabel("Reaction coordinate s")
ax.set_ylabel("Energy (eV)")
ax.set_title("Minimum Energy Path")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Top-right: Path evolution ---
ax = axes[0, 1]
ax.plot(s_dense, E_dense, "-", color="#94a3b8", linewidth=1.0, alpha=0.6)
n_show = min(len(path_history), 8)
cmap = plt.cm.viridis
for j in range(n_show):
    alpha = 0.3 + 0.7 * (j / max(n_show - 1, 1))
    color = cmap(j / max(n_show - 1, 1))
    if j < len(energy_history):
        E_j = energy_history[j]
    else:
        E_j_raw = compute_energies(path_history[j])
        E_j = fill_missing_energies(path_history[j], E_j_raw)
    ax.plot(
        path_history[j], E_j,
        "o-", color=color, alpha=alpha, markersize=4, linewidth=1.0,
        label=f"Iter {j}" if j in [0, n_show - 1] else None
    )
ax.set_xlabel("Reaction coordinate s")
ax.set_ylabel("Energy (eV)")
ax.set_title("Path Evolution")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Bottom-left: Convergence ---
ax = axes[1, 0]
iters = list(range(len(max_force_history)))
ax.semilogy(iters, max_force_history, "o-", color="#8b5cf6", markersize=5, linewidth=2)
ax.axhline(y=FORCE_TOL, color="#ef4444", linestyle="--", linewidth=1, label=f"Tolerance ({FORCE_TOL})")
ax.set_xlabel("NEB iteration")
ax.set_ylabel("Max |F$_{NEB}$| (eV/Å)")
ax.set_title("Force Convergence")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Bottom-right: Barrier comparison ---
ax = axes[1, 1]

# NEB barrier
neb_barrier = E_ts_neb - E_final[0]

# Exact barrier
exact_barrier = pes_energy(0.0) - pes_energy(S_REACTANT)

# Direct scan: compute energies at dense points via uniqx
s_scan = np.linspace(S_REACTANT, S_PRODUCT, 21)
E_scan_raw = compute_energies(s_scan)
E_scan = fill_missing_energies(s_scan, E_scan_raw)
scan_barrier = max(E_scan) - E_scan[0]

labels = ["NEB\n(optimised)", "Direct scan\n(21 points)", "Exact\n(analytical)"]
barriers = [neb_barrier, scan_barrier, exact_barrier]
colors = ["#2563eb", "#16a34a", "#94a3b8"]
bars = ax.bar(range(3), barriers, color=colors, alpha=0.85, edgecolor="#1e293b", linewidth=0.8)
for bar, val in zip(bars, barriers):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
        f"{val:.4f} eV", ha="center", va="bottom", fontsize=9, fontweight="bold"
    )
ax.set_xticks(range(3))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Barrier height (eV)")
ax.set_title("Barrier Height Comparison")
ax.grid(axis="y", alpha=0.3)

fig.suptitle(
    "NEB Reaction Path: Cl$^-$ + CH$_3$Cl → ClCH$_3$ + Cl$^-$",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

## 11. Annotated Reaction Path Diagram

A publication-quality diagram showing the converged MEP with annotated
reactant, product, and transition state regions.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Background shading for reactant and product wells
ax.axvspan(S_REACTANT - 0.5, -0.5, color="#dbeafe", alpha=0.5, label="Reactant well")
ax.axvspan(0.5, S_PRODUCT + 0.5, color="#dcfce7", alpha=0.5, label="Product well")
ax.axvspan(-0.5, 0.5, color="#fef3c7", alpha=0.5, label="TS region")

# True PES
ax.plot(s_dense, E_dense, "-", color="#94a3b8", linewidth=2, label="PES: $E(s) = as^4 - bs^2 + c$")

# Converged MEP
ax.plot(
    current_images, E_final,
    "s-", color="#2563eb", markersize=9, linewidth=2.5, label="NEB MEP", zorder=4
)

# TS marker
ax.plot(s_ts, E_ts_neb, "*", color="#f59e0b", markersize=22, zorder=5)
ax.annotate(
    f"Transition State\n$E_a$ = {barrier_fwd:.3f} eV\ns = {s_ts:+.3f}",
    xy=(s_ts, E_ts_neb),
    xytext=(s_ts + 0.8, E_ts_neb + 0.3),
    fontsize=10, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#f59e0b", linewidth=2),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="#fffbeb", edgecolor="#f59e0b"),
)

# Barrier height annotation
ax.annotate(
    "", xy=(S_REACTANT, E_ts_neb), xytext=(S_REACTANT, E_final[0]),
    arrowprops=dict(arrowstyle="<->", color="#dc2626", linewidth=1.5)
)
ax.text(
    S_REACTANT - 0.15, (E_ts_neb + E_final[0]) / 2,
    f"$E_a$ = {barrier_fwd:.3f} eV", fontsize=9, color="#dc2626",
    ha="right", va="center",
)

# Reactant/Product labels
ax.text(S_REACTANT, E_final[0] - 0.15, "Cl$^-$ + CH$_3$Cl", fontsize=10, ha="center", fontweight="bold")
ax.text(S_PRODUCT, E_final[-1] - 0.15, "ClCH$_3$ + Cl$^-$", fontsize=10, ha="center", fontweight="bold")

ax.set_xlabel("Reaction coordinate s", fontsize=12)
ax.set_ylabel("Energy (eV)", fontsize=12)
ax.set_title(
    "S$_N$2 Reaction: Nudged Elastic Band Minimum Energy Path",
    fontsize=14, fontweight="bold"
)
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Results Summary

In [ ]:
print("NEB Reaction Path Results")
print("=" * 60)
print()
print("Configuration:")
print(f"  Reaction:       Cl- + CH3Cl -> ClCH3 + Cl- (SN2)")
print(f"  PES model:      E(s) = {PES_A}*s^4 - {PES_B}*s^2 + {PES_C}")
print(f"  Images:         {N_IMAGES} movable + 2 fixed = {n_total} total")
print(f"  Qubits:         {N_QUBITS} (dim = {2**N_QUBITS})")
print(f"  Spring constant: {K_SPRING} eV/A^2")
print(f"  Step size:      {STEP_SIZE}")
print(f"  Hardware:       {rec['label']}")
print()
print("Transition State:")
print(f"  Position (NEB):   s = {s_ts:+.4f}")
print(f"  Position (exact): s = 0.0000")
print(f"  Energy (NEB):     {E_ts_neb:.4f} eV")
print(f"  Energy (exact):   {pes_energy(0.0):.4f} eV")
print(f"  TS position err:  {abs(s_ts):.4f}")
print()
print("Barrier Height:")
print(f"  Forward (NEB):    {barrier_fwd:.4f} eV")
print(f"  Reverse (NEB):    {barrier_rev:.4f} eV")
print(f"  Exact:            {exact_barrier:.4f} eV")
print(f"  Error:            {abs(barrier_fwd - exact_barrier):.4f} eV")
print()
print("Convergence:")
print(f"  Iterations:       {len(max_force_history) - 1}")
print(f"  Final max |F|:    {max_force_history[-1]:.4f} eV/A")
print(f"  Converged:        {'Yes' if converged else 'No'}")
print(f"  Total time:       {dt_total:.2f}s")
print(f"  Eigensolves:      {n_submits}")

## Summary

| Aspect | Detail |
|:-------|:-------|
| **Problem** | S$_N$2 reaction path via Nudged Elastic Band |
| **PES model** | Double-well $E(s) = as^4 - bs^2 + c$, symmetric around $s=0$ |
| **Method** | NEB with projected forces + harmonic springs |
| **uniqx op** | `eigs(H(s), k=1)` per image per iteration |
| **Transition state** | Saddle point at $s \approx 0$, barrier $\approx E_a$ |
| **Hardware** | Cost model routes each eigensolve to optimal backend |

The NEB method successfully locates the transition state and converges
to the minimum energy path. Each image's energy comes from a genuine
eigensolve on uniqx — the same infrastructure that handles
production-scale molecular Hamiltonians. Scaling to larger active spaces
(more qubits) would automatically trigger GPU or QPU routing via the
cost model.

## 10. Comparison: NEB Eigensolve vs Analytical PES

The SN2 double-well potential is analytically known, so we can validate the
NEB uniqx eigensolve results against the analytical $E(s) = as^4 - bs^2 + c$.

We compare:
1. **Barrier height** from NEB vs the analytical barrier
2. **MEP energies** at each image vs the exact PES
3. **Convergence quality** -- how close is the converged path to the true MEP
4. **Timing** for the NEB iteration loop

In [ ]:
%%time

# --- Validate NEB results against analytical PES ---

# Exact energies at the converged image positions
E_exact_at_images = [pes_energy(s) for s in current_images]
E_errors = [abs(e_gw - e_ex) for e_gw, e_ex in zip(E_final, E_exact_at_images)]

# Analytical barrier
exact_barrier = pes_energy(0.0) - pes_energy(S_REACTANT)
neb_barrier = E_ts_neb - E_final[0]

print("NEB vs Analytical PES Comparison:")
print(f"  {'Image':>5} {'s':>8} {'E_NEB':>10} {'E_exact':>10} {'|error|':>10}")
print(f"  {'-' * 5} {'-' * 8} {'-' * 10} {'-' * 10} {'-' * 10}")
for i, (s, e_neb, e_ex, err) in enumerate(
    zip(current_images, E_final, E_exact_at_images, E_errors)
):
    tag = " <-- TS" if i == np.argmax(E_final) else ""
    print(f"  {i:>5} {s:>+8.3f} {e_neb:>10.4f} {e_ex:>10.4f} {err:>10.2e}{tag}")

print(f"\nBarrier Height:")
print(f"  NEB barrier:       {neb_barrier:.4f} eV")
print(f"  Analytical barrier: {exact_barrier:.4f} eV")
print(f"  |error|:           {abs(neb_barrier - exact_barrier):.4f} eV ({abs(neb_barrier - exact_barrier) / exact_barrier * 100:.1f}%)")
print(f"\nMEP Quality:")
print(f"  Max |E_NEB - E_exact|:  {max(E_errors):.2e} eV")
print(f"  Mean |E_NEB - E_exact|: {np.mean(E_errors):.2e} eV")

# Dense analytical PES for overlay plot
s_dense = np.linspace(S_REACTANT, S_PRODUCT, 200)
E_dense = [pes_energy(s) for s in s_dense]

# --- Numpy eigensolve validation ---
# Verify the uniqx eigensolve vs direct numpy at each image position
numpy_energies_at_images = []
for s in current_images:
    H, dim = build_reaction_hamiltonian(s, N_QUBITS)
    H_np = np.array(H)
    evals = np.linalg.eigvalsh(H_np)
    numpy_energies_at_images.append(evals[0])

numpy_vs_gw = [abs(n - g) for n, g in zip(numpy_energies_at_images, E_final)]
print(f"\nGateway vs Numpy eigensolve:")
print(f"  Max |dE|:  {max(numpy_vs_gw):.2e} eV")
print(f"  Mean |dE|: {np.mean(numpy_vs_gw):.2e} eV")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Left: MEP overlay with analytical PES ---
axes[0].plot(s_dense, E_dense, "r-", linewidth=2.0, label="Analytical PES", alpha=0.7)
axes[0].plot(current_images, E_final, "bs-", markersize=7, linewidth=2.0, label="NEB (uniqx)")
axes[0].plot(current_images, numpy_energies_at_images, "g^--", markersize=5, linewidth=1.0, label="Numpy eigvalsh")
axes[0].axhline(y=pes_energy(0.0), color="gray", linestyle=":", alpha=0.4)
axes[0].annotate(
    f"TS: E={pes_energy(0.0):.3f} eV",
    xy=(0.0, pes_energy(0.0)),
    xytext=(0.4, pes_energy(0.0) + 0.1),
    arrowprops=dict(arrowstyle="->", color="gray"),
    fontsize=9,
)
axes[0].set_xlabel("Reaction coordinate s")
axes[0].set_ylabel("Energy (eV)")
axes[0].set_title("MEP: NEB vs Analytical PES")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# --- Middle: Energy error at each image ---
axes[1].bar(range(len(E_errors)), E_errors, color="#dc2626", alpha=0.7, edgecolor="black")
axes[1].set_xlabel("Image index")
axes[1].set_ylabel("|E_NEB - E_exact| (eV)")
axes[1].set_title("Energy Error per Image")
axes[1].grid(axis="y", alpha=0.3)

# --- Right: Barrier height comparison ---
bar_labels = ["Analytical\nbarrier", "NEB\nbarrier"]
bar_vals = [exact_barrier, neb_barrier]
bar_colors = ["#16a34a", "#2563eb"]
axes[2].bar(bar_labels, bar_vals, color=bar_colors, alpha=0.8, edgecolor="black")
axes[2].set_ylabel("Barrier height (eV)")
axes[2].set_title("Barrier: NEB vs Analytical")
axes[2].grid(axis="y", alpha=0.3)
for i, v in enumerate(bar_vals):
    axes[2].text(i, v + 0.01, f"{v:.4f} eV", ha="center", fontsize=10)

# Add error annotation
err_pct = abs(neb_barrier - exact_barrier) / exact_barrier * 100
axes[2].annotate(
    f"Error: {err_pct:.1f}%",
    xy=(0.5, min(bar_vals) * 0.8),
    fontsize=11, ha="center", color="#dc2626",
)

fig.suptitle(
    "NEB Reaction Path: uniqx Eigensolve vs Analytical PES",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.show()